<a href="https://colab.research.google.com/github/Maham411/Hite/blob/main/CSX_Net_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSX-Net — reproduction notebook (HITE 2026 camera-ready revision)

**Paper:** *CSX-Net: A Hybrid CNN–Self-Attention-LSTM–XGBoost Ensemble for Robust Per-Class Intrusion Detection in Digital Twin-Enabled IoT Networks*

This one notebook reproduces every table and figure in the revised paper and runs the experiments the reviewers asked for.

| Section | What it produces | Paper item | Approx. time on a T4 GPU |
|---|---|---|---|
| 1–4 | Setup, dataset download, 533,998-record sample, Table I | Table I | 10–15 min (download + read) |
| 5 | Figures 2–4 redrawn from the published numbers; McNemar effect sizes | Figs. 2–4, Table IV | < 1 min (CPU) |
| 6 | **Out-of-fold (5-fold) stacking for one seed** + SALSTM baseline + ablation rows | Table VI, Sec. VI-F | ~1.5 h per seed |
| 7 | Seed-42 extras: SALSTM+SMOTE, column-permutation test, base-paper subset, TCP/UDP neighbour analysis, CPU cost | Secs. VI-C, VI-D, VI-E, VI-G | ~1 h |
| 8 | Optional: original five configurations (in-sample stacking, SMOTE) | Tables II–V, Figs. 2–4 from your own run | ~1.8 h |
| 9 | Optional: device-wise hold-out | (future work in paper) | ~1.5 h |
| 10 | Aggregate all seeds → `results_macros.tex` for Overleaf | all new numbers | < 1 min |

**Recommended 6-hour plan (two free GPUs in parallel)**
* **Colab:** `SEED = 42`, `RUN_OOF = True`, `RUN_EXTRAS = True` → about 2.5 h.
* **Kaggle (or a second Colab account):** `SEED = 1`, `RUN_OOF = True`, `RUN_EXTRAS = False` → about 1.5 h. Add `SEED = 2` afterwards if time allows.
* Copy the `results/seed_1/` (and `seed_2/`) folders into the Colab `OUT_DIR/results/`, then run Section 10 once.

Set `SMOKE_TEST = True` to check the whole pipeline in about 3 minutes on synthetic data before the real run.

## 1. Configuration

In [2]:
# ======================= EDIT THESE =======================
SEED          = 42      # 42 on Colab; 1 (and 2) on the second machine
RUN_OOF       = False    # Section 6: 5-fold out-of-fold stacking for this SEED
RUN_EXTRAS    = False   # Section 7: only needed once, with SEED = 42
RUN_ORIGINAL  = False   # Section 8: re-run the original in-sample experiments (Tables II-V)
RUN_DEVICE_HOLDOUT = False  # Section 9: optional device-wise hold-out
SMOKE_TEST    = False   # True = tiny synthetic dataset, 1 epoch: checks the code end to end
USE_DRIVE     = True    # Colab: save outputs to Google Drive so a disconnect loses nothing
CODE_URL      = "https://github.com/USERNAME/CSX-Net"   # put your public repository URL here
# ==========================================================

SAMPLE_FRAC   = 0.07561   # per-file sampling fraction used in the paper
SAMPLE_SEED   = 42        # the 533,998-record sample is fixed; SEED only changes split/folds/init
N_FOLDS       = 5
EPOCHS        = 15
BATCH         = 128
VAL_FRAC      = 0.10
XGB_PARAMS    = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=1.0,
                     objective="multi:softprob", eval_metric="mlogloss", tree_method="hist")
# NOTE: the paper does not list XGBoost hyper-parameters; if your original run used different
# values, put them here so the in-sample reproduction (Section 8) matches Tables II-V.

if SMOKE_TEST:
    EPOCHS, SAMPLE_FRAC = 1, 1.0

## 2. Environment

In [3]:
import os, sys, json, time, glob, re, subprocess, shutil, platform, warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
try:
    import xgboost, imblearn
except ImportError:
    _pip("xgboost", "imbalanced-learn")

if IN_COLAB and USE_DRIVE and not SMOKE_TEST:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/CSX-Net"
elif IN_KAGGLE:
    OUT_DIR = "/kaggle/working/CSX-Net"
else:
    OUT_DIR = os.path.abspath("CSX-Net-out" if not SMOKE_TEST else "CSX-Net-smoke")
DATA_DIR = os.path.join(OUT_DIR, "data") if not IN_KAGGLE else "/kaggle/working/data"
RES_DIR  = os.path.join(OUT_DIR, "results")
FIG_DIR  = os.path.join(OUT_DIR, "figures")
for d in (DATA_DIR, RES_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
print("Outputs ->", OUT_DIR)

import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
import tensorflow as tf
tf.get_logger().setLevel("ERROR")
import keras
from keras import layers
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             balanced_accuracy_score, confusion_matrix)
from scipy.stats import chi2 as chi2_dist
import matplotlib
matplotlib.use("Agg") if not (IN_COLAB or IN_KAGGLE) else None
import matplotlib.pyplot as plt
import seaborn as sns

GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    tf.config.experimental.set_memory_growth(g, True)
XGB_PARAMS["device"] = "cuda" if GPUS else "cpu"
print("TensorFlow", tf.__version__, "| XGBoost", xgb.__version__, "| GPU:", [g.name for g in GPUS] or "none")
if not GPUS and not SMOKE_TEST:
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU. CPU-only training is ~10x slower.")

CLASSES = ['Ack', 'Benign', 'Combo', 'Junk', 'SYN', 'Scan', 'TCP', 'UDP', 'UDP Plain']
C2I = {c: i for i, c in enumerate(CLASSES)}
TCP, UDP = C2I['TCP'], C2I['UDP']

Mounted at /content/drive
Outputs -> /content/drive/MyDrive/CSX-Net
TensorFlow 2.21.0 | XGBoost 3.4.1 | GPU: ['/physical_device:GPU:0']


## 3. Dataset: download N-BaIoT
Source: UCI Machine Learning Repository, dataset 442 (CC BY 4.0), about 1.7 GB.
The archive holds one folder per device with `benign_traffic.csv`, `gafgyt_attacks.rar` and (for 7 of 9 devices) `mirai_attacks.rar`.
On Kaggle you can instead attach any N-BaIoT copy that contains the 89 CSV files (e.g. named `1.benign.csv`, `1.gafgyt.tcp.csv`, `1.mirai.udpplain.csv`) and set `RAW_DIR` to it; the loader understands both layouts.

In [4]:
UCI_URL = "https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip"
RAW_DIR = os.path.join(DATA_DIR, "nbaiot_raw")      # change if you attached the data elsewhere

def make_synthetic_dataset(root, rows=60, seed=0):
    """SMOKE_TEST only: 89 small CSV files in the Kaggle naming scheme, 115 columns."""
    rng = np.random.default_rng(seed)
    cols = [f"f{i}" for i in range(115)]
    os.makedirs(root, exist_ok=True)
    g_att, m_att = ["combo", "junk", "scan", "tcp", "udp"], ["ack", "scan", "syn", "udp", "udpplain"]
    for dev in range(1, 10):
        files = [("benign", None)] + [("gafgyt", a) for a in g_att]
        if dev not in (3, 7):
            files += [("mirai", a) for a in m_att]
        for fam, att in files:
            key = (fam, att)
            centre = (hash(key) % 97) / 97.0
            X = rng.normal(centre, 0.3, size=(rows, 115)) + 0.05 * dev
            name = f"{dev}.benign.csv" if fam == "benign" else f"{dev}.{fam}.{att}.csv"
            pd.DataFrame(X, columns=cols).to_csv(os.path.join(root, name), index=False)

def download_and_extract_uci(raw_dir):
    os.makedirs(raw_dir, exist_ok=True)
    zpath = os.path.join(DATA_DIR, "nbaiot.zip")
    if not os.path.exists(zpath):
        print("Downloading", UCI_URL)
        subprocess.run(["wget", "-q", "-O", zpath, UCI_URL], check=True)
    subprocess.run(["unzip", "-q", "-n", zpath, "-d", raw_dir], check=True)
    if shutil.which("unrar") is None:
        subprocess.run("apt-get -qq install -y unrar >/dev/null 2>&1", shell=True)
    for rar in glob.glob(os.path.join(raw_dir, "**", "*.rar"), recursive=True):
        dest = os.path.join(os.path.dirname(rar), os.path.basename(rar).split("_")[0])  # .../gafgyt or .../mirai
        if not os.path.isdir(dest) or not os.listdir(dest):
            os.makedirs(dest, exist_ok=True)
            subprocess.run(["unrar", "x", "-o+", "-inul", rar, dest + os.sep], check=True)

if SMOKE_TEST:
    RAW_DIR = os.path.join(DATA_DIR, "synthetic")
    make_synthetic_dataset(RAW_DIR)
elif not glob.glob(os.path.join(RAW_DIR, "**", "*.csv"), recursive=True):
    download_and_extract_uci(RAW_DIR)
print(len(glob.glob(os.path.join(RAW_DIR, "**", "*.csv"), recursive=True)), "CSV files under", RAW_DIR)

88 CSV files under /content/drive/MyDrive/CSX-Net/data/nbaiot_raw


## 4. Map the 89 files to nine labels and draw the 533,998-record sample (Table I)

In [5]:
LABEL_MAP = {('benign', None): 'Benign',
             ('mirai', 'ack'): 'Ack', ('mirai', 'scan'): 'Scan', ('gafgyt', 'scan'): 'Scan',
             ('mirai', 'syn'): 'SYN', ('mirai', 'udp'): 'UDP', ('gafgyt', 'udp'): 'UDP',
             ('mirai', 'udpplain'): 'UDP Plain', ('gafgyt', 'combo'): 'Combo',
             ('gafgyt', 'junk'): 'Junk', ('gafgyt', 'tcp'): 'TCP'}
SKIP = {"features", "device_info", "data_summary", "readme", "demonstrate_structure"}

def parse_file(path, root):
    """Return (family, attack, device) from either the UCI folder layout or the N.family.attack.csv layout."""
    stem = os.path.basename(path)[:-4].lower()
    if any(s in stem for s in SKIP):
        return None
    rel = os.path.relpath(path, root).lower().replace("\\", "/")
    parts = stem.split(".")
    if parts[0].isdigit():                       # Kaggle style: 1.gafgyt.tcp
        device = parts[0]
        if parts[1] == "benign":
            return ("benign", None, device)
        return (parts[1], parts[2], device)
    device = rel.split("/")[-2] if "/" in rel else "unknown"
    if "benign" in stem:
        device = rel.split("/")[-2]
        return ("benign", None, device)
    fam = "mirai" if "mirai" in rel else ("gafgyt" if ("gafgyt" in rel or "bashlite" in rel) else None)
    if fam is None:
        return None
    segs = rel.split("/")
    device = segs[-3] if len(segs) >= 3 else "unknown"
    return (fam, stem.replace("mirai-", ""), device)

def load_sample(root, frac=SAMPLE_FRAC, seed=SAMPLE_SEED, benign_pool_frac=0.10):
    frames, pool, n_raw, n_files, header = [], [], 0, 0, None
    for path in sorted(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)):
        info = parse_file(path, root)
        if info is None:
            continue
        fam, att, dev = info
        label = LABEL_MAP.get((fam, att))
        if label is None:
            print("  skipped (no label):", path); continue
        df = pd.read_csv(path)
        df = df.astype("float32")
        header = header or list(df.columns)
        n_raw += len(df); n_files += 1
        s = df.sample(frac=frac, random_state=seed)
        s = s.assign(label=label, family=fam, device=str(dev))
        frames.append(s)
        if fam == "benign":       # extra Benign pool for the base-paper-subset experiment (needs 49,433 Benign)
            p = df.sample(frac=benign_pool_frac, random_state=seed + 1)
            pool.append(p.assign(label=label, family=fam, device=str(dev)))
    data = pd.concat(frames, ignore_index=True)
    pool = pd.concat(pool, ignore_index=True) if pool else data.iloc[:0]
    print(f"{n_files} files matched, {n_raw:,} raw records -> sample of {len(data):,}")
    return data, pool, header

data_pkl = os.path.join(DATA_DIR, "sample.pkl")
if os.path.exists(data_pkl) and not SMOKE_TEST:
    data, benign_pool, HEADER = pd.read_pickle(data_pkl), pd.read_pickle(data_pkl + ".pool"), json.load(open(data_pkl + ".hdr"))
else:
    data, benign_pool, HEADER = load_sample(RAW_DIR)
    if not SMOKE_TEST:
        data.to_pickle(data_pkl); benign_pool.to_pickle(data_pkl + ".pool"); json.dump(HEADER, open(data_pkl + ".hdr", "w"))
FEATS = [c for c in data.columns if c not in ("label", "family", "device")]
assert len(FEATS) == 115, len(FEATS)
print("First 12 feature columns (column order used by the CNN, Sec. IV-B):", HEADER[:12])

BASE_COUNTS = {'UDP': 3377, 'TCP': 3390, 'Scan': 3277, 'SYN': 2260, 'Ack': 3370,
               'Benign': 49433, 'UDP Plain': 3383, 'Combo': 3307, 'Junk': 3367}
cnt = data["label"].value_counts()
table1 = pd.DataFrame({"Base paper [5]": pd.Series(BASE_COUNTS), "This study": cnt,
                       "Share (%)": (100 * cnt / cnt.sum()).round(2)}).sort_values("This study", ascending=False)
table1.loc["Total"] = [table1["Base paper [5]"].sum(), table1["This study"].sum(), 100.0]
table1.to_csv(os.path.join(RES_DIR, "table1_class_distribution.csv"))
print(table1)

First 12 feature columns (column order used by the CNN, Sec. IV-B): ['MI_dir_L5_weight', 'MI_dir_L5_mean', 'MI_dir_L5_variance', 'MI_dir_L3_weight', 'MI_dir_L3_mean', 'MI_dir_L3_variance', 'MI_dir_L1_weight', 'MI_dir_L1_mean', 'MI_dir_L1_variance', 'MI_dir_L0.1_weight', 'MI_dir_L0.1_mean', 'MI_dir_L0.1_variance']
           Base paper [5]  This study  Share (%)
UDP                3377.0    164553.0      30.82
TCP                3390.0     65013.0      12.17
Scan               3277.0     59965.0      11.23
SYN                2260.0     55446.0      10.38
Ack                3370.0     48678.0       9.12
Benign            49433.0     42033.0       7.87
UDP Plain          3383.0     39568.0       7.41
Combo              3307.0     38950.0       7.29
Junk               3367.0     19794.0       3.71
Total             75164.0    534000.0     100.00


In [6]:
   key = pd.util.hash_pandas_object(data[FEATS].round(6), index=False)
   g = pd.DataFrame({"key": key.values, "label": data["label"].values})
   multi = g.groupby("key")["label"].nunique(); ck = multi[multi > 1].index
   print("rows sharing an identical feature vector with another label:", g["key"].isin(ck).sum())
   print(g[g["key"].isin(ck)]["label"].value_counts())
   tr, te = train_test_split(data, test_size=0.30, stratify=data["label"], random_state=42)
   ktr = set(pd.util.hash_pandas_object(tr[FEATS].round(6), index=False))
   kte = pd.util.hash_pandas_object(te[FEATS].round(6), index=False)
   print("test rows with an exact copy in train:", kte.isin(ktr).sum(), "of", len(te))

rows sharing an identical feature vector with another label: 73022
label
UDP    71484
TCP     1538
Name: count, dtype: int64
test rows with an exact copy in train: 55041 of 160200


## 5. Helpers: preprocessing, models, metrics, and the published figures

In [8]:
def split_and_scale(df, seed, test_size=0.30):
    """Stratified 70/30 split; imputer and min-max scaler fitted on the training split only."""
    tr, te = train_test_split(df, test_size=test_size, stratify=df["label"], random_state=seed)
    imp, sc = SimpleImputer(strategy="mean"), MinMaxScaler()
    Xtr = sc.fit_transform(imp.fit_transform(tr[FEATS].values)).astype("float32")
    Xte = np.clip(sc.transform(imp.transform(te[FEATS].values)), 0, 1).astype("float32")
    ytr = tr["label"].map(C2I).values.astype("int64"); yte = te["label"].map(C2I).values.astype("int64")
    return Xtr, ytr, Xte, yte, tr.reset_index(drop=True), te.reset_index(drop=True)

@keras.saving.register_keras_serializable(package="csxnet")
class SelfAttention(layers.Layer):
    """SAV = sum_t gamma_t J_t, gamma = softmax over learned similarity scores (Eq. 1)."""
    def build(self, input_shape):
        d = int(input_shape[-1])
        self.W = self.add_weight(name="W", shape=(d, d), initializer="glorot_uniform")
        self.b = self.add_weight(name="b", shape=(d,), initializer="zeros")
        self.u = self.add_weight(name="u", shape=(d, 1), initializer="glorot_uniform")
    def call(self, J):
        score = keras.ops.matmul(keras.ops.tanh(keras.ops.matmul(J, self.W) + self.b), self.u)
        gamma = keras.ops.softmax(score, axis=1)
        return keras.ops.sum(gamma * J, axis=1)

def build_cnn(n_feat=115):
    i = keras.Input((n_feat, 1))
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(i)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(32, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    return keras.Model(i, layers.Dense(9, activation="softmax")(x), name="cnn_branch")

def build_salstm(n_feat=115):
    i = keras.Input((n_feat, 1))
    x = layers.LSTM(128, return_sequences=True)(i)
    x = layers.LSTM(64, return_sequences=True)(x)
    x = layers.LSTM(32, return_sequences=True)(x)
    x = SelfAttention()(x)
    return keras.Model(i, layers.Dense(9, activation="softmax")(x), name="salstm")

def build_gru(n_feat=115):
    # The base paper does not specify its GRU baseline; we mirror the SALSTM depth without attention.
    i = keras.Input((n_feat, 1))
    x = layers.GRU(128, return_sequences=True)(i)
    x = layers.GRU(64, return_sequences=True)(x)
    x = layers.GRU(32)(x)
    return keras.Model(i, layers.Dense(9, activation="softmax")(x), name="gru")

BUILDERS = {"cnn": build_cnn, "salstm": build_salstm, "gru": build_gru}

def fit_nn(kind, X, y, seed, verbose=0):
    """Adam 1e-3, categorical cross-entropy, 15 epochs, batch 128, 10% validation.
    The data are shuffled BEFORE the validation hold-out, so the validation set is a random 10%
    (Keras' validation_split takes the last rows, which after SMOTE are all synthetic)."""
    keras.utils.set_random_seed(seed)
    idx = np.random.default_rng(seed).permutation(len(X))
    n_val = int(VAL_FRAC * len(X)); va, tr = idx[:n_val], idx[n_val:]
    m = BUILDERS[kind]()
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    h = m.fit(X[tr, :, None], y[tr], validation_data=(X[va, :, None], y[va]), epochs=EPOCHS,
              batch_size=BATCH, verbose=verbose, shuffle=True)
    return m, time.time() - t0, float(h.history["val_accuracy"][-1])

def predict_proba(m, X, bs=4096):
    return m.predict(X[:, :, None], batch_size=bs, verbose=0)

def fit_xgb(Z, y, seed):
    clf = xgb.XGBClassifier(num_class=9, random_state=seed, **XGB_PARAMS)
    clf.fit(Z, y)
    return clf

def metrics(y, p):
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=range(9), zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y, p, average="weighted", zero_division=0)
    return {"acc": 100 * accuracy_score(y, p), "w_prec": 100 * wp, "w_rec": 100 * wr, "w_f1": 100 * wf,
            "macro_f1": 100 * f1.mean(), "bal_acc": 100 * balanced_accuracy_score(y, p),
            "tcp_f1": 100 * f1[TCP], "per_class": {"prec": (100 * pr).tolist(), "rec": (100 * rc).tolist(),
                                                     "f1": (100 * f1).tolist(), "support": sup.tolist()}}

def mcnemar(y, pa, pb):
    """Continuity-corrected McNemar; b = only A correct, c = only B correct; paired 95% CI on dAcc."""
    a, b_ = (pa == y), (pb == y)
    b, c = int(np.sum(a & ~b_)), int(np.sum(~a & b_)); n = len(y)
    chi2 = (abs(b - c) - 1) ** 2 / (b + c) if (b + c) else 0.0
    d = (b - c) / n; se = np.sqrt((b + c) / n ** 2 - (b - c) ** 2 / n ** 3)
    return {"chi2": chi2, "p": float(chi2_dist.sf(chi2, 1)), "b": b, "c": c, "disc": b + c,
            "b_over_c": b / c if c else float("inf"), "dacc": 100 * d,
            "ci": (100 * (d - 1.96 * se), 100 * (d + 1.96 * se))}

def bootstrap_ci(y, p, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); n = len(y); accs, mf1 = [], []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        cm = np.bincount(y[i] * 9 + p[i], minlength=81).reshape(9, 9)
        tp = np.diag(cm); prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1)
        f1 = np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0)
        accs.append(100 * tp.sum() / n); mf1.append(100 * f1.mean())
    q = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return {"acc": q(accs), "macro_f1": q(mf1)}

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1, default=float)

In [9]:
# ---- Published results (Tables II-IV, Fig. 4) used to redraw the paper's figures exactly ----
import numpy as np
CLASSES = ['Ack','Benign','Combo','Junk','SYN','Scan','TCP','UDP','UDP Plain']
TABLE2 = {  # accuracy, weighted precision, recall, F1 (%)
    'Naive Bayes': [57.71, 49.40, 57.71, 48.76],
    'GRU': [87.18, 84.83, 87.18, 82.10],
    'SALSTM (baseline, reproduced)': [87.07, 85.07, 87.07, 82.00],
    'CSX-Net (proposed)': [99.28, 99.30, 99.28, 99.28],
    'CSX-Net (no-SMOTE ablation)': [99.22, 99.23, 99.22, 99.21],
}
F1_SALSTM = [99.13, 99.62, 96.95, 93.58, 99.88, 99.75, 0.11, 83.26, 99.11]
F1_CSX    = [99.96, 99.92, 99.95, 99.92, 99.99, 99.97, 97.13, 98.89, 99.89]
CM_SALSTM = np.array([[14438,0,0,0,0,0,0,85,80],[0,12558,0,0,1,35,7,6,3],[0,8,11650,11,15,0,0,1,0],
    [0,3,698,5231,4,2,0,0,0],[0,0,1,0,16621,10,0,2,0],[0,17,0,0,8,17959,0,6,0],[2,8,0,0,0,1,11,19482,0],
    [68,8,0,0,0,9,2,49175,104],[19,0,0,0,0,2,0,4,11845]])
CM_CSX = np.array([[14600,0,0,1,0,0,0,1,1],[0,12601,0,0,0,2,5,1,1],[0,0,11680,3,2,0,0,0,0],
    [0,0,4,5934,0,0,0,0,0],[0,0,0,0,16633,1,0,0,0],[0,6,0,0,0,17983,0,1,0],[0,0,1,0,0,1,18423,1079,0],
    [4,6,1,1,0,0,1,49344,9],[6,0,0,0,0,1,0,8,11855]])
MCNEMAR = {  # chi2 (continuity-corrected), discordant pairs, as published in Table IV
    'CSX-Net vs. SALSTM': (19505.18, 19623), 'CSX-Net vs. GRU': (19327.23, 19461),
    'CSX-Net vs. Naive Bayes': (64479.03, 68799), 'CSX-Net (SMOTE) vs. no SMOTE': (67.15, 158)}
N_TEST = 160200

In [10]:
# ---- Figures 2-4 (same content and matplotlib/seaborn style as the paper) ----
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

FIG_RC = {"font.size": 13, "axes.titlesize": 14, "axes.labelsize": 13,
          "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11}

def fig2_overall(table2, path_png, path_pdf=None):
    """table2: dict model -> [acc, prec, rec, f1] (percent). Order = paper legend order."""
    with plt.rc_context(FIG_RC):
        metrics = ["Accuracy", "Precision", "Recall", "F1-score"]
        models = list(table2.keys())
        x = np.arange(len(metrics)); w = 0.8 / len(models)
        fig, ax = plt.subplots(figsize=(8, 3.4))
        for i, m in enumerate(models):
            ax.bar(x - 0.4 + w / 2 + i * w, table2[m], w, label=m)
        ax.set_xticks(x); ax.set_xticklabels(metrics)
        ax.set_ylabel("Percentage (%)"); ax.set_ylim(0, 132); ax.set_yticks(range(0, 101, 20))
        ax.set_title("Overall Performance Comparison on N-BaIoT")
        ax.legend(loc="upper center", ncol=3, framealpha=0.95, fontsize=10)
        fig.tight_layout(); fig.savefig(path_png, dpi=300)
        if path_pdf: fig.savefig(path_pdf)
        plt.close(fig)

def fig3_perclass_f1(classes, f1_csx, f1_sal, path_png, path_pdf=None):
    with plt.rc_context(FIG_RC):
        x = np.arange(len(classes)); w = 0.38
        fig, ax = plt.subplots(figsize=(10, 3.6))
        ax.bar(x - w / 2, f1_csx, w, label="CSX-Net (proposed)")
        ax.bar(x + w / 2, f1_sal, w, label="SALSTM (baseline)")
        ax.set_xticks(x); ax.set_xticklabels(classes)
        ax.set_xlabel("Class"); ax.set_ylabel("F1-score (%)"); ax.set_ylim(0, 150); ax.set_yticks(range(0, 101, 20))
        ax.set_title("Per-Class F1-score: SALSTM (baseline) vs CSX-Net (proposed)")
        ax.legend(title="Model", loc="upper center", ncol=2, framealpha=0.95)
        fig.tight_layout(); fig.savefig(path_png, dpi=300)
        if path_pdf: fig.savefig(path_pdf)
        plt.close(fig)

def fig4_confusion(classes, cm_sal, cm_csx, path_png, path_pdf=None):
    with plt.rc_context(FIG_RC):
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        for ax, cm, cmap, title in [(axes[0], cm_sal, "Blues", "SALSTM (baseline) — Confusion Matrix"),
                                    (axes[1], cm_csx, "Greens", "CSX-Net (proposed) — Confusion Matrix")]:
            sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax, xticklabels=classes,
                        yticklabels=classes, annot_kws={"size": 11})
            ax.set_title(title); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        fig.tight_layout(); fig.savefig(path_png, dpi=200)
        if path_pdf: fig.savefig(path_pdf)
        plt.close(fig)


# Redraw Figures 2-4 from the published numbers (identical to the paper)
fig2_overall(TABLE2, os.path.join(FIG_DIR, "fig2_overall.png"), os.path.join(FIG_DIR, "fig2_overall.pdf"))
fig3_perclass_f1(CLASSES, F1_CSX, F1_SALSTM, os.path.join(FIG_DIR, "fig3_perclass_f1.png"), os.path.join(FIG_DIR, "fig3_perclass_f1.pdf"))
fig4_confusion(CLASSES, CM_SALSTM, CM_CSX, os.path.join(FIG_DIR, "fig4_confusion.png"), os.path.join(FIG_DIR, "fig4_confusion.pdf"))

# Table IV effect sizes recovered exactly from the published chi2 and discordant counts
rows = []
for name, (chi2, disc) in MCNEMAR.items():
    d = np.sqrt(chi2 * disc) + 1; b, c = (disc + d) / 2, (disc - d) / 2
    se = np.sqrt(disc - d * d / N_TEST) / N_TEST
    rows.append([name, chi2, disc, round(b), round(c), round(b / c, 2), round(100 * d / N_TEST, 3),
                 round(100 * (d / N_TEST - 1.96 * se), 3), round(100 * (d / N_TEST + 1.96 * se), 3)])
table4 = pd.DataFrame(rows, columns=["Comparison", "chi2", "disc", "b", "c", "b/c", "dAcc", "CI_lo", "CI_hi"])
table4.to_csv(os.path.join(RES_DIR, "table4_mcnemar_effect_sizes.csv"), index=False)
print(table4.to_string(index=False))
# macro-F1 / balanced accuracy for Table III's macro row
for nm, cm in [("SALSTM", CM_SALSTM), ("CSX-Net", CM_CSX)]:
    tp = np.diag(cm); pr = tp / cm.sum(0); rc = tp / cm.sum(1); f1 = 2 * pr * rc / (pr + rc)
    print(f"{nm}: macro-P {100*pr.mean():.2f}  macro-R (bal. acc.) {100*rc.mean():.2f}  macro-F1 {100*f1.mean():.2f}")

                  Comparison     chi2  disc     b    c    b/c   dAcc  CI_lo  CI_hi
          CSX-Net vs. SALSTM 19505.18 19623 19594   29 675.67 12.213 12.052 12.373
             CSX-Net vs. GRU 19327.23 19461 19428   33 588.72 12.107 11.947 12.267
     CSX-Net vs. Naive Bayes 64479.03 68799 67702 1097  61.72 41.576 41.328 41.824
CSX-Net (SMOTE) vs. no SMOTE    67.15   158   131   27   4.85  0.065  0.050  0.080
SALSTM: macro-P 90.85  macro-R (bal. acc.) 87.27  macro-F1 85.71
CSX-Net: macro-P 99.71  macro-R (bal. acc.) 99.34  macro-F1 99.51


## 6. Out-of-fold stacking for one seed (Table VI)
Five stratified folds of the training split; both branches are trained on four folds and predict the fifth (out-of-fold meta-features). XGBoost is trained on those OOF posteriors only; test posteriors are averaged over the five fold models. Every fold is saved to `OUT_DIR/results/seed_<SEED>/` as soon as it finishes, so a disconnect resumes where it stopped.

In [11]:
def run_oof(seed):
    sd = os.path.join(RES_DIR, f"seed_{seed}"); os.makedirs(sd, exist_ok=True)
    Xtr, ytr, Xte, yte, tr_df, te_df = split_and_scale(data, seed)
    print(f"seed {seed}: train {len(ytr):,}  test {len(yte):,}")
    oof = {k: np.zeros((len(ytr), 9), "float32") for k in ("cnn", "salstm")}
    tst = {k: np.zeros((len(yte), 9), "float32") for k in ("cnn", "salstm")}
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)
    for k, (a, b) in enumerate(skf.split(Xtr, ytr)):
        fpath = os.path.join(sd, f"fold{k}.npz")
        if os.path.exists(fpath):
            z = np.load(fpath)
            for br in ("cnn", "salstm"):
                oof[br][b] = z[f"oof_{br}"]; tst[br] += z[f"test_{br}"] / N_FOLDS
            print(f"  fold {k}: loaded"); continue
        out, info = {}, {}
        for br in ("cnn", "salstm"):
            m, secs, va = fit_nn(br, Xtr[a], ytr[a], seed + 100 * k + (0 if br == "cnn" else 1))
            out[f"oof_{br}"] = predict_proba(m, Xtr[b]); out[f"test_{br}"] = predict_proba(m, Xte)
            info[br] = {"train_s": secs, "val_acc": va}
            if k == 0 and seed == 42:
                m.save(os.path.join(sd, f"fold0_{br}.keras"))
            keras.backend.clear_session()
        np.savez_compressed(fpath, **out); save_json(info, fpath + ".json")
        for br in ("cnn", "salstm"):
            oof[br][b] = out[f"oof_{br}"]; tst[br] += out[f"test_{br}"] / N_FOLDS
        print(f"  fold {k}: done  {info}")

    res = {}
    # full stack and ablations
    xgb_full = fit_xgb(np.hstack([oof["cnn"], oof["salstm"]]), ytr, seed)
    p_csx = xgb_full.predict(np.hstack([tst["cnn"], tst["salstm"]]))
    xgb_sal = fit_xgb(oof["salstm"], ytr, seed)
    preds = {"CSX_OOF": p_csx, "SALSTM_XGB": xgb_sal.predict(tst["salstm"]),
             "CNN_branch": tst["cnn"].argmax(1), "SALSTM_branch": tst["salstm"].argmax(1),
             "SoftVote": (tst["cnn"] + tst["salstm"]).argmax(1)}
    if seed == 42:
        xgb_full.save_model(os.path.join(sd, "xgb_full.json"))
    # SALSTM baseline trained on the full training split (as in Tables II-V)
    bpath = os.path.join(sd, "salstm_full.npz")
    if os.path.exists(bpath):
        preds["SALSTM_full"] = np.load(bpath)["pred"]
    else:
        m, secs, _ = fit_nn("salstm", Xtr, ytr, seed + 7)
        preds["SALSTM_full"] = predict_proba(m, Xte).argmax(1); keras.backend.clear_session()
        np.savez_compressed(bpath, pred=preds["SALSTM_full"])
    for name, p in preds.items():
        res[name] = metrics(yte, p)
    res["_mcnemar_CSX_vs_SALSTM"] = mcnemar(yte, p_csx, preds["SALSTM_full"])
    if seed == 42:
        res["_bootstrap_CSX"] = bootstrap_ci(yte, p_csx)
    np.savez_compressed(os.path.join(sd, "test_predictions.npz"), y=yte, **preds)
    save_json(res, os.path.join(sd, "summary.json"))
    print(pd.DataFrame({k: {m_: v[m_] for m_ in ("acc", "macro_f1", "bal_acc", "tcp_f1")}
                        for k, v in res.items() if not k.startswith("_")}).T.round(2))
    return res

if RUN_OOF:
    oof_res = run_oof(SEED)

In [12]:
# 6b. Duplicate-free evaluation
key = pd.util.hash_pandas_object(data[FEATS].round(6), index=False).values
d = data.assign(_k=key)
d = d[d.groupby("_k")["label"].transform("nunique") == 1]          # drop TCP/UDP-identical vectors
d = d.drop_duplicates("_k").drop(columns="_k").reset_index(drop=True)  # keep one copy of each record
print(f"{len(data):,} -> {len(d):,} records after de-duplication")
counts = d["label"].value_counts(); print(counts)

if counts.min() < 50:
    print("WARNING: a class has fewer than 50 unique records - send me these counts before training.")
else:
    data_full, RES_full = data, RES_DIR
    data, RES_DIR = d, os.path.join(RES_full, "dedup"); os.makedirs(RES_DIR, exist_ok=True)
    try:
        dedup_res = run_oof(42)          # same 5-fold OOF pipeline, now on duplicate-free data
    finally:
        data, RES_DIR = data_full, RES_full

534,000 -> 363,805 records after de-duplication
label
UDP          88389
Scan         53358
SYN          52481
Ack          46021
Benign       41654
UDP Plain    37520
Combo        29350
Junk         14977
TCP             55
Name: count, dtype: int64
seed 42: train 254,663  test 109,142
  fold 0: done  {'cnn': {'train_s': 103.74878978729248, 'val_acc': 0.8122024536132812}, 'salstm': {'train_s': 436.76465010643005, 'val_acc': 0.939969539642334}}
  fold 1: done  {'cnn': {'train_s': 86.77216076850891, 'val_acc': 0.8945663571357727}, 'salstm': {'train_s': 458.4035859107971, 'val_acc': 0.9873361587524414}}
  fold 2: done  {'cnn': {'train_s': 100.69186305999756, 'val_acc': 0.845187246799469}, 'salstm': {'train_s': 451.3549346923828, 'val_acc': 0.975261390209198}}
  fold 3: done  {'cnn': {'train_s': 91.16279196739197, 'val_acc': 0.8267805576324463}, 'salstm': {'train_s': 451.1164243221283, 'val_acc': 0.9597015380859375}}
  fold 4: done  {'cnn': {'train_s': 91.98572778701782, 'val_acc': 0.7932

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [20:14:45] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


                 acc  macro_f1  bal_acc  tcp_f1
CSX_OOF        96.75     85.81    86.10     0.0
SALSTM_XGB     95.52     85.06    85.47     0.0
CNN_branch     86.85     74.44    74.36     0.0
SALSTM_branch  99.04     87.88    88.01     0.0
SoftVote       98.54     87.43    87.52     0.0
SALSTM_full    98.82     87.34    87.00     0.0


In [7]:
te_dup = kte.isin(ktr).values
print(pd.crosstab(te["label"].values, te_dup, normalize="index").round(3))
P = np.load(os.path.join(RES_DIR, "seed_42", "test_predictions.npz"))
for k in ["SALSTM_full", "SALSTM_branch", "CSX_OOF"]:
    m = metrics(P["y"][~te_dup], P[k][~te_dup])
    print(k, round(m["acc"], 2), round(m["macro_f1"], 2), round(m["tcp_f1"], 2))

col_0      False  True 
row_0                  
Ack        0.922  0.078
Benign     0.989  0.011
Combo      0.660  0.340
Junk       0.658  0.342
SYN        0.925  0.075
Scan       0.848  0.152
TCP        0.000  1.000
UDP        0.526  0.474
UDP Plain  0.925  0.075


NameError: name 'metrics' is not defined

## 7. Seed-42 extras (run once, with `SEED = 42`)
SALSTM trained with SMOTE (factorial design, R1-7) · column-permutation test for the CNN (R1-6, R2-4) · base-paper-like 75,164-record subset (R1-2, R3-3) · nearest-neighbour family of the residual TCP→UDP errors (R3-5) · CPU-only cost and parameter count (R3-7, R2).

In [ ]:
def run_extras():
    ed = os.path.join(RES_DIR, "extras"); os.makedirs(ed, exist_ok=True)
    out = json.load(open(os.path.join(ed, "extras.json"))) if os.path.exists(os.path.join(ed, "extras.json")) else {}
    Xtr, ytr, Xte, yte, tr_df, te_df = split_and_scale(data, 42)

    if "salstm_smote" not in out:            # (a) SALSTM + SMOTE
        Xs, ys = SMOTE(random_state=42).fit_resample(Xtr, ytr)
        print("SMOTE training set:", len(ys))
        m, secs, _ = fit_nn("salstm", Xs.astype("float32"), ys, 42)
        mt = metrics(yte, predict_proba(m, Xte).argmax(1)); keras.backend.clear_session()
        out["salstm_smote"] = {"acc": mt["acc"], "macro_f1": mt["macro_f1"], "bal_acc": mt["bal_acc"],
                               "tcp_f1": mt["tcp_f1"], "train_s": secs}
        save_json(out, os.path.join(ed, "extras.json")); print("SALSTM+SMOTE:", out["salstm_smote"])

    if "permutation" not in out:             # (b) CNN with original vs permuted column order
        accs = []
        for r, perm in enumerate([np.arange(115)] + [np.random.default_rng(1000 + j).permutation(115) for j in range(3)]):
            m, _, _ = fit_nn("cnn", Xtr[:, perm], ytr, 42)
            accs.append(metrics(yte, predict_proba(m, Xte[:, perm]).argmax(1))["acc"]); keras.backend.clear_session()
        out["permutation"] = {"orig_acc": accs[0], "perm_accs": accs[1:]}
        save_json(out, os.path.join(ed, "extras.json")); print("Permutation test:", out["permutation"])

    if "subset" not in out:                  # (c) base-paper-like class counts
        parts = []
        for cls, n in BASE_COUNTS.items():
            src = benign_pool if cls == "Benign" and len(benign_pool) else data[data["label"] == cls]
            src = src[src["label"] == cls]
            parts.append(src.sample(n=min(n, len(src)), random_state=42))
        sub = pd.concat(parts, ignore_index=True)
        sXtr, sytr, sXte, syte, _, _ = split_and_scale(sub, 42)
        m, _, _ = fit_nn("salstm", sXtr, sytr, 42)
        mt = metrics(syte, predict_proba(m, sXte).argmax(1)); keras.backend.clear_session()
        out["subset"] = {"n": len(sub), "acc": mt["acc"], "tcp_f1": mt["tcp_f1"], "macro_f1": mt["macro_f1"]}
        save_json(out, os.path.join(ed, "extras.json")); print("Base-paper subset:", out["subset"])

    pred_file = os.path.join(RES_DIR, "seed_42", "test_predictions.npz")
    if "tcp_nn" not in out and os.path.exists(pred_file):     # (d) TCP->UDP errors: nearest UDP family
        P = np.load(pred_file); assert np.array_equal(P["y"], yte)
        err = np.where((yte == TCP) & (P["CSX_OOF"] == UDP))[0]
        udp_tr = np.where(ytr == UDP)[0]
        fam = tr_df["family"].values[udp_tr]
        if len(err):
            nn = NearestNeighbors(n_neighbors=1).fit(Xtr[udp_tr])
            _, j = nn.kneighbors(Xte[err])
            pct = 100 * np.mean(fam[j[:, 0]] == "gafgyt")
        else:
            pct = float("nan")
        out["tcp_nn"] = {"n_err": int(len(err)), "pct_gafgyt": float(pct), "base_gafgyt": float(100 * np.mean(fam == "gafgyt"))}
        save_json(out, os.path.join(ed, "extras.json")); print("TCP/UDP neighbours:", out["tcp_nn"])

    f0 = os.path.join(RES_DIR, "seed_42", "fold0_cnn.keras")
    if "cpu" not in out and os.path.exists(f0):              # (e) CPU-only inference cost
        with tf.device("/CPU:0"):
            cnn = keras.models.load_model(f0); sal = keras.models.load_model(f0.replace("cnn", "salstm"))
            clf = xgb.XGBClassifier(); clf.load_model(os.path.join(RES_DIR, "seed_42", "xgb_full.json"))
            clf.set_params(device="cpu")
            Xb = Xte[:20000]
            predict_proba(cnn, Xb[:512]); predict_proba(sal, Xb[:512])   # warm-up
            t0 = time.time()
            z = np.hstack([predict_proba(cnn, Xb, 1024), predict_proba(sal, Xb, 1024)]); clf.predict(z)
            ms = 1000 * (time.time() - t0) / len(Xb)
        pc, ps = cnn.count_params(), sal.count_params()
        out["cpu"] = {"n_cpu": os.cpu_count(), "ms_per_sample": ms, "params_cnn": int(pc), "params_sal": int(ps),
                      "mb": 4 * (pc + ps) / 2 ** 20}
        save_json(out, os.path.join(ed, "extras.json")); print("CPU cost:", out["cpu"])
    return out

if RUN_EXTRAS:
    extras = run_extras()

## 8. Optional: original experiments (Tables II–V, Figs. 2–4 from your own run)
In-sample stacking exactly as in Tables II–V (SMOTE on the training split; meta-features predicted on the training split). GPU non-determinism means the numbers will be close to, not identical with, the published ones; the paper keeps the published values.

In [ ]:
def run_original():
    od = os.path.join(RES_DIR, "original"); os.makedirs(od, exist_ok=True)
    Xtr, ytr, Xte, yte, _, _ = split_and_scale(data, 42)
    preds, times = {}, {}
    t0 = time.time(); nb = GaussianNB().fit(Xtr, ytr); times["Naive Bayes"] = time.time() - t0
    preds["Naive Bayes"] = nb.predict(Xte)
    for kind, name in [("gru", "GRU"), ("salstm", "SALSTM (baseline, reproduced)")]:
        m, secs, _ = fit_nn(kind, Xtr, ytr, 42); times[name] = secs
        preds[name] = predict_proba(m, Xte).argmax(1); keras.backend.clear_session()
    for use_smote, name in [(True, "CSX-Net (proposed)"), (False, "CSX-Net (no-SMOTE ablation)")]:
        X, y = (SMOTE(random_state=42).fit_resample(Xtr, ytr) if use_smote else (Xtr, ytr))
        X = X.astype("float32"); t0 = time.time()
        mc, _, _ = fit_nn("cnn", X, y, 42); ms_, _, _ = fit_nn("salstm", X, y, 43)
        Ztr = np.hstack([predict_proba(mc, X), predict_proba(ms_, X)])          # in-sample meta-features
        clf = fit_xgb(Ztr, y, 42); times[name] = time.time() - t0
        preds[name] = clf.predict(np.hstack([predict_proba(mc, Xte), predict_proba(ms_, Xte)]))
        keras.backend.clear_session()
    M = {k: metrics(yte, p) for k, p in preds.items()}
    t2 = pd.DataFrame({k: [v["acc"], v["w_prec"], v["w_rec"], v["w_f1"]] for k, v in M.items()},
                      index=["Accuracy", "Precision", "Recall", "F1-score"]).T.round(2)
    t2.to_csv(os.path.join(od, "table2_overall.csv")); print(t2)
    csx, sal = "CSX-Net (proposed)", "SALSTM (baseline, reproduced)"
    t3 = pd.DataFrame({"Support": M[sal]["per_class"]["support"],
                       **{f"SALSTM {k}": M[sal]["per_class"][k] for k in ("prec", "rec", "f1")},
                       **{f"CSX {k}": M[csx]["per_class"][k] for k in ("prec", "rec", "f1")}}, index=CLASSES).round(2)
    t3.to_csv(os.path.join(od, "table3_perclass.csv")); print(t3)
    t4 = {f"CSX-Net vs. {b}": mcnemar(yte, preds[csx], preds[b]) for b in ("SALSTM (baseline, reproduced)", "GRU", "Naive Bayes")}
    t4["CSX-Net (SMOTE) vs. no SMOTE"] = mcnemar(yte, preds[csx], preds["CSX-Net (no-SMOTE ablation)"])
    pd.DataFrame(t4).T.to_csv(os.path.join(od, "table4_mcnemar.csv")); print(pd.DataFrame(t4).T)
    pd.Series(times, name="train_s").to_csv(os.path.join(od, "table5_train_time.csv"))
    order = ["Naive Bayes", "GRU", "SALSTM (baseline, reproduced)", "CSX-Net (proposed)", "CSX-Net (no-SMOTE ablation)"]
    fig2_overall({k: t2.loc[k].tolist() for k in order}, os.path.join(od, "fig2_overall_rerun.png"))
    fig3_perclass_f1(CLASSES, M[csx]["per_class"]["f1"], M[sal]["per_class"]["f1"], os.path.join(od, "fig3_perclass_f1_rerun.png"))
    fig4_confusion(CLASSES, confusion_matrix(yte, preds[sal], labels=range(9)),
                   confusion_matrix(yte, preds[csx], labels=range(9)), os.path.join(od, "fig4_confusion_rerun.png"))
    save_json({k: {m_: v[m_] for m_ in ("acc", "w_f1", "macro_f1", "bal_acc", "tcp_f1")} for k, v in M.items()},
              os.path.join(od, "summary.json"))

if RUN_ORIGINAL:
    run_original()

## 9. Optional: device-wise hold-out (R2-2)
Trains on seven devices and tests on two unseen devices that carry both Mirai and Gafgyt traffic. Not reported in the paper (listed as future work); results go to `results/device_holdout/summary.json`.

In [ ]:
def run_device_holdout(test_devices=None):
    dd = os.path.join(RES_DIR, "device_holdout"); os.makedirs(dd, exist_ok=True)
    devs = sorted(data["device"].unique())
    with_mirai = sorted(data.loc[data["family"] == "mirai", "device"].unique())
    test_devices = test_devices or with_mirai[:2]
    print("test devices:", test_devices)
    tr = data[~data["device"].isin(test_devices)]; te = data[data["device"].isin(test_devices)]
    imp, sc = SimpleImputer(), MinMaxScaler()
    Xtr = sc.fit_transform(imp.fit_transform(tr[FEATS].values)).astype("float32")
    Xte = np.clip(sc.transform(imp.transform(te[FEATS].values)), 0, 1).astype("float32")
    ytr, yte = tr["label"].map(C2I).values, te["label"].map(C2I).values
    oof = np.zeros((len(ytr), 18), "float32"); tst = np.zeros((len(yte), 18), "float32")
    for k, (a, b) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=42).split(Xtr, ytr)):
        for j, br in enumerate(("cnn", "salstm")):
            m, _, _ = fit_nn(br, Xtr[a], ytr[a], 42 + k)
            oof[b, 9 * j:9 * j + 9] = predict_proba(m, Xtr[b]); tst[:, 9 * j:9 * j + 9] += predict_proba(m, Xte) / N_FOLDS
            keras.backend.clear_session()
    p = fit_xgb(oof, ytr, 42).predict(tst)
    m, _, _ = fit_nn("salstm", Xtr, ytr, 42); ps = predict_proba(m, Xte).argmax(1)
    res = {"test_devices": list(test_devices), "CSX_OOF": metrics(yte, p), "SALSTM": metrics(yte, ps)}
    save_json(res, os.path.join(dd, "summary.json"))
    print({k: (v["acc"], v["macro_f1"]) for k, v in res.items() if k != "test_devices"})

if RUN_DEVICE_HOLDOUT:
    run_device_holdout()

## 10. Aggregate all seeds and write `results_macros.tex` for Overleaf
Run this after every seed folder you want to report is present in `OUT_DIR/results/` (copy `seed_1/`, `seed_2/` from the other machine). Upload the generated `results/results_macros.tex` to Overleaf, replacing the placeholder file. Read any WARNING lines printed below: they flag a sentence in the paper whose wording must be checked against your results.

In [ ]:
# ---- Section 9: write results/results_macros.tex for the paper ----------------
import json, glob, os
import numpy as np

def _ms(vals, nd=2):
    """'mean$\\pm$sd' string (sd omitted when only one value)."""
    v = np.asarray(vals, float)
    if len(v) == 1:
        return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f}{{\\scriptsize$\\pm${v.std(ddof=1):.{nd}f}}}"

def _m(vals):
    return float(np.mean(vals))

def write_latex_macros(summary, out_path, code_url="https://github.com/USERNAME/CSX-Net"):
    """summary: dict produced by aggregate_results(); see Section 8."""
    S = summary
    seeds = S["seeds"]
    rows = S["rows"]           # name -> dict(metric -> list over seeds)
    def cell(key, met):
        if key not in rows or met not in rows[key] or len(rows[key][met]) == 0:
            return "--"
        return _ms(rows[key][met])
    names = [("SALSTM_full", "SALSTM (full training split)"),
             ("SALSTM_SMOTE", "SALSTM + SMOTE$^\\dagger$"),
             ("CNN_branch", "CNN branch only"),
             ("SALSTM_branch", "SALSTM branch only"),
             ("SoftVote", "CNN + SALSTM soft vote"),
             ("SALSTM_XGB", "SALSTM $\\rightarrow$ XGBoost"),
             ("CSX_OOF", "\\textbf{CSX-Net (OOF)}")]
    body = ""
    for k, label in names:
        body += f"{label} & {cell(k,'acc')} & {cell(k,'macro_f1')} & {cell(k,'bal_acc')} & {cell(k,'tcp_f1')}\\\\\n"

    csx = rows["CSX_OOF"]
    acc, mac, tcp = _m(csx["acc"]), _m(csx["macro_f1"]), _m(csx["tcp_f1"])
    ci = S.get("bootstrap_seed42")
    # ---- generated sentences (wording chosen from the measured values) ----
    d = acc - 99.22
    if abs(d) <= 0.30:
        oof = (f"This is within {abs(d):.2f} points of the in-sample no-SMOTE result (99.22\\%), "
               "so the in-sample meta-features did not inflate the headline.")
    elif d < 0:
        oof = (f"This is {abs(d):.2f} points below the in-sample no-SMOTE result (99.22\\%); the in-sample "
               "construction inflated the headline by that margin, and the OOF figure is the one we stand behind.")
    else:
        oof = (f"This is {d:.2f} points above the in-sample no-SMOTE result (99.22\\%), "
               "so the in-sample meta-features did not inflate the headline.")

    cnn_t, sal_t = _m(rows["CNN_branch"]["tcp_f1"]), _m(rows["SALSTM_branch"]["tcp_f1"])
    sv_t, sx_t = _m(rows["SoftVote"]["tcp_f1"]), _m(rows["SALSTM_XGB"]["tcp_f1"])
    cnn_needed = (tcp - sx_t) >= 2.0
    abl = (f"Alone, the CNN and SALSTM branches reach TCP F1 of {cnn_t:.2f}\\% and {sal_t:.2f}\\%, and soft voting "
           f"{sv_t:.2f}\\%; ")
    if cnn_needed:
        abl += (f"XGBoost on SALSTM posteriors alone gives {sx_t:.2f}\\%, so the CNN adds "
                f"{tcp - sx_t:.2f} points beyond re-partitioning the SALSTM output.")
    else:
        abl += (f"XGBoost on SALSTM posteriors alone already gives {sx_t:.2f}\\%, within "
                f"{abs(tcp - sx_t):.2f} points of CSX-Net, so most of the repair comes from the meta-learner "
                "re-partitioning SALSTM posteriors, not from the CNN.")

    def _opt(key):
        return S.get(key) or None
    P = _opt("permutation") or {"orig_acc": float("nan"), "perm_accs": [float("nan")]}
    drop = P["orig_acc"] - np.mean(P["perm_accs"])
    adjacency = drop >= 0.5
    perm = (f"Randomly permuting the 115 columns (three draws) moves CNN-only accuracy from "
            f"{P['orig_acc']:.2f}\\% to {_ms(P['perm_accs'])}\\%, ")
    perm += ("so the column layout carries information the branch uses." if adjacency else
             "so the branch does not depend on adjacency and is better read as a complementary learned representation.")

    if cnn_needed and adjacency:
        concl = ("The complementary error profiles of a convolutional and a recurrent branch are what make the "
                 "fusion effective, and the CNN's gain depends on the grouped column layout.")
    elif cnn_needed:
        concl = ("The complementary error profiles of a convolutional and a recurrent branch make the fusion "
                 "effective, although the CNN's contribution does not hinge on column adjacency.")
    else:
        concl = ("The ablation locates most of the repair in the gradient-boosted meta-learner re-partitioning "
                 "SALSTM posteriors, with the CNN branch adding a smaller complementary share.")

    Q = _opt("salstm_smote") or {"acc": float("nan"), "tcp_f1": float("nan")}
    smote = (f"SALSTM trained on the SMOTE-balanced split (seed 42) reaches {Q['acc']:.2f}\\% accuracy and "
             f"{Q['tcp_f1']:.2f}\\% TCP F1, ")
    smote += ("so resampling alone does not repair the single model, completing the factorial design."
              if Q["tcp_f1"] < 50 else
              "so resampling also helps the single model, and on it the two effects are not fully separable.")

    B = _opt("subset") or {"acc": float("nan"), "tcp_f1": float("nan")}
    sub = (f"Retraining SALSTM on a 75,164-record subsample matching the base paper's class counts "
           f"gives {B['acc']:.2f}\\% accuracy ({B['tcp_f1']:.2f}\\% TCP F1), ")
    sub += ("close to the reported figure, supporting this explanation." if B["acc"] >= 97 else
            "so sampling explains only part of the gap.")

    T = _opt("tcp_nn") or {"n_err": 0, "pct_gafgyt": float("nan"), "base_gafgyt": float("nan")}
    tcpnn = (f"For the {T['n_err']:,} TCP records the OOF model (seed 42) still labels UDP, the nearest UDP "
             f"training record is Gafgyt in {T['pct_gafgyt']:.1f}\\% of cases (base rate {T['base_gafgyt']:.1f}\\%).")

    C = _opt("cpu") or {"n_cpu": 0, "ms_per_sample": float("nan"), "params_cnn": 0, "params_sal": 0, "mb": float("nan")}
    cost = (f"On CPU alone ({C['n_cpu']} vCPUs) CSX-Net needs {C['ms_per_sample']:.3f}~ms per sample with "
            f"{C['params_cnn'] + C['params_sal']:,} branch parameters ({C['mb']:.2f}~MB), within reach of an edge gateway.")

    PEND = "\\pend{not run}"
    if not _opt("permutation"): perm = PEND
    if not _opt("salstm_smote"): smote = PEND
    if not _opt("subset"): sub = PEND
    if not _opt("tcp_nn"): tcpnn = PEND
    if not _opt("cpu"): cost = PEND
    ci = ci or {"acc": (float("nan"),) * 2, "macro_f1": (float("nan"),) * 2}
    n = len(seeds)
    words = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five"}
    seedlist = ", ".join(str(s) for s in seeds[:-1]) + (" and " if n > 1 else "") + str(seeds[-1])
    tex = f"""% AUTO-GENERATED by the CSX-Net notebook -- do not edit by hand
\\newcommand{{\\pend}}[1]{{\\textcolor{{red}}{{[#1]}}}}
\\newcommand{{\\NSeeds}}{{{words.get(n, str(n))}}}
\\newcommand{{\\SeedList}}{{{seedlist}}}
\\newcommand{{\\CodeURL}}{{{code_url}}}
\\newcommand{{\\CSXOOFAcc}}{{{acc:.2f}}}
\\newcommand{{\\CSXOOFAccCI}}{{[{ci['acc'][0]:.2f}, {ci['acc'][1]:.2f}]}}
\\newcommand{{\\CSXOOFMacro}}{{{mac:.2f}}}
\\newcommand{{\\CSXOOFMacroCI}}{{[{ci['macro_f1'][0]:.2f}, {ci['macro_f1'][1]:.2f}]}}
\\newcommand{{\\CSXOOFTCP}}{{{tcp:.2f}}}
\\newcommand{{\\OOFSentence}}{{{oof}}}
\\newcommand{{\\AblationSentence}}{{{abl}}}
\\newcommand{{\\PermSentence}}{{{perm}}}
\\newcommand{{\\SALSmoteSentence}}{{{smote}}}
\\newcommand{{\\SubsetSentence}}{{{sub}}}
\\newcommand{{\\TCPNNSentence}}{{{tcpnn}}}
\\newcommand{{\\CostSentence}}{{{cost}}}
\\newcommand{{\\ConclusionMechanism}}{{{concl}}}
\\newcommand{{\\TabVIRows}}{{%
{body}}}
"""
    with open(out_path, "w") as f:
        f.write(tex)
    # Warnings that require a manual text check in the paper
    warn = []
    if _opt("salstm_smote") and Q["tcp_f1"] >= 50:
        warn.append("SALSTM+SMOTE repairs TCP: revise the RQ2 paragraph (Sec. VI-D) and the conclusion's SMOTE paragraph.")
    if not cnn_needed:
        warn.append("CNN adds <2 TCP-F1 points over SALSTM->XGBoost: soften 'We attribute the repair to the branches' (Sec. VI-B).")
    if d < -0.30:
        warn.append("OOF accuracy is >0.3 points below 99.22: consider leading the abstract with the OOF figure.")
    return warn


def aggregate_results():
    seeds = sorted(int(os.path.basename(d).split("_")[1]) for d in glob.glob(os.path.join(RES_DIR, "seed_*"))
                   if os.path.exists(os.path.join(d, "summary.json")))
    assert seeds, "No finished seed found: run Section 6 first."
    seeds = sorted(seeds, key=lambda s: (s != 42, s))
    keys = ["SALSTM_full", "CNN_branch", "SALSTM_branch", "SoftVote", "SALSTM_XGB", "CSX_OOF"]
    rows = {k: {m: [] for m in ("acc", "macro_f1", "bal_acc", "tcp_f1")} for k in keys}
    boot = None
    for s in seeds:
        R = json.load(open(os.path.join(RES_DIR, f"seed_{s}", "summary.json")))
        for k in keys:
            for m in rows[k]:
                rows[k][m].append(R[k][m])
        if s == 42:
            boot = R.get("_bootstrap_CSX")
    ex_path = os.path.join(RES_DIR, "extras", "extras.json")
    ex = json.load(open(ex_path)) if os.path.exists(ex_path) else {}
    if "salstm_smote" in ex:
        q = ex["salstm_smote"]; rows["SALSTM_SMOTE"] = {m: [q[m]] for m in ("acc", "macro_f1", "bal_acc", "tcp_f1")}
    summary = {"seeds": seeds, "rows": rows, "bootstrap_seed42": boot, "permutation": ex.get("permutation"),
               "salstm_smote": ex.get("salstm_smote"), "subset": ex.get("subset"),
               "tcp_nn": ex.get("tcp_nn"), "cpu": ex.get("cpu")}
    save_json(summary, os.path.join(RES_DIR, "summary_all_seeds.json"))
    tab6 = pd.DataFrame({k: {m: f"{np.mean(v):.2f} ± {np.std(v, ddof=1) if len(v) > 1 else 0:.2f}" for m, v in r.items()}
                         for k, r in rows.items()}).T
    tab6.to_csv(os.path.join(RES_DIR, "table6_oof_ablation.csv")); print(tab6)
    return summary

summary = aggregate_results()
warnings = write_latex_macros(summary, os.path.join(RES_DIR, "results_macros.tex"), CODE_URL)
print(open(os.path.join(RES_DIR, "results_macros.tex")).read())
for w in warnings:
    print("WARNING:", w)
print("Done. Upload", os.path.join(RES_DIR, "results_macros.tex"), "to Overleaf (replace results_macros.tex).")